In [27]:
#Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

In [28]:
# Load + combine
df = pd.concat([
    pd.read_csv("../data/processed_22.csv"),
    pd.read_csv("../data/processed_23.csv"),
    pd.read_csv("../data/processed_24.csv"),
    pd.read_csv("../data/processed_25.csv")
], ignore_index=True)
df["date"] = pd.to_datetime(df["date"])

In [29]:
# Target: daily expected Lingulodinium polyedra per mL
y = df["Lpoly_expected_ml"].astype(float)
X = df.drop(columns=["date", "Lpoly_expected", "Lpoly_expected_ml"]).dropna(axis=1, how="all")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def eval_reg(name, model):
    model.fit(X_train, y_train)
    pred_test = model.predict(X_test)
    return {
        "Model": name,
        "Test_R2": float(r2_score(y_test, pred_test)),
        "Test_RMSE": rmse(y_test, pred_test),
    }


In [30]:
# Baselines
lin = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
ridge = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])

rf = RandomForestRegressor(
    n_estimators=80,
    max_depth=10,
    random_state=42,
    n_jobs=1
)

perf = pd.DataFrame([
    eval_reg("Linear", lin),
    eval_reg("Ridge", ridge),
    eval_reg("RandomForest", rf),
]).sort_values("Test_R2", ascending=False)

print("Baseline regression performance (random split)")
print(perf)


Baseline regression performance (random split)
          Model   Test_R2  Test_RMSE
2  RandomForest  0.577215   7.521570
1         Ridge  0.349270   9.331446
0        Linear  0.244977  10.051450


In [31]:
# Feature importance

# 1) Ridge coefficients (standardized) = interpretable linear influence
ridge.fit(X_train, y_train)
coef = ridge.named_steps["model"].coef_
ridge_importance = pd.DataFrame({
    "feature": X.columns,
    "ridge_coef_abs": np.abs(coef),
    "ridge_coef_signed": coef
}).sort_values("ridge_coef_abs", ascending=False).reset_index(drop=True)

print("\nTop 20 Ridge coefficients (|coef|, standardized features)")
print(ridge_importance.head(20))

# 2) Random Forest impurity importance = nonlinear importance signal
rf.fit(X_train, y_train)
rf_importance = pd.DataFrame({
    "feature": X.columns,
    "rf_impurity_importance": rf.feature_importances_
}).sort_values("rf_impurity_importance", ascending=False).reset_index(drop=True)

print("\nTop 20 RandomForest impurity importances")
print(rf_importance.head(20))


Top 20 Ridge coefficients (|coef|, standardized features)
                                 feature  ridge_coef_abs  ridge_coef_signed
0                                 Ring05        9.908269          -9.908269
1                   texture_third_moment        8.435433          -8.435433
2                                  HOG34        8.150373           8.150373
3             texture_average_gray_level        7.655359           7.655359
4              RotatedBoundingBox_xwidth        7.108316           7.108316
5                                  HOG17        7.059257           7.059257
6   summedConvexPerimeter_over_Perimeter        7.047513           7.047513
7                                  HOG32        6.896620           6.896620
8                             ROI_per_ml        6.761929           6.761929
9                                  HOG52        6.077542           6.077542
10                                 HOG41        5.565478           5.565478
11                           

In [32]:
#Top 13 biological features only

# Explicitly define effort / metadata features to exclude
effort_features = ["ml_analyzed", "roiCount", "ROI_per_ml"]

top13_bio = [
    f for f in rf_importance["feature"].tolist()
    if f not in effort_features
][:13]

print("\nTop 13 biological features:")
print(top13_bio)

X_train_top13_bio = X_train[top13_bio]
X_test_top13_bio = X_test[top13_bio]

rf_top13_bio = RandomForestRegressor(
    n_estimators=80,
    max_depth=10,
    random_state=42,
    n_jobs=1
)

rf_top13_bio.fit(X_train_top13_bio, y_train)

#Results
y_pred_top13_bio = rf_top13_bio.predict(X_test_top13_bio)

r2_top13_bio = r2_score(y_test, y_pred_top13_bio)
rmse_top13_bio = np.sqrt(np.mean((y_test - y_pred_top13_bio) ** 2))

print("\nReduced top-13 BIOLOGICAL RF performance")
print(f"Test R2:   {r2_top13_bio:.3f}")
print(f"Test RMSE: {rmse_top13_bio:.3f}")

rf_top13_bio_importance = pd.DataFrame({
    "feature": top13_bio,
    "rf_impurity_importance": rf_top13_bio.feature_importances_
}).sort_values("rf_impurity_importance", ascending=False)

print("\nReduced RF feature importance (biological only)")
print(rf_top13_bio_importance)



Top 13 biological features:
['moment_invariant1', 'Ring10', 'moment_invariant2', 'moment_invariant3', 'Ring09', 'HOG79', 'moment_invariant7', 'HOG41', 'HOG81', 'HOG10', 'Perimeter', 'HOG42', 'moment_invariant4']

Reduced top-13 BIOLOGICAL RF performance
Test R2:   0.523
Test RMSE: 7.990

Reduced RF feature importance (biological only)
              feature  rf_impurity_importance
1              Ring10                0.208303
0   moment_invariant1                0.200698
6   moment_invariant7                0.093944
4              Ring09                0.093060
7               HOG41                0.083212
2   moment_invariant2                0.081065
5               HOG79                0.055422
3   moment_invariant3                0.051083
8               HOG81                0.040302
10          Perimeter                0.038959
12  moment_invariant4                0.021509
11              HOG42                0.016647
9               HOG10                0.015797


In [ ]:
df_all = pd.concat([
    pd.read_csv("../data/processed_22.csv"),7
    pd.read_csv("../data/processed_23.csv"),
    pd.read_csv("../data/processed_24.csv"),
    pd.read_csv("../data/processed_25.csv")
], ignore_index=True)

df_all["date"] = pd.to_datetime(df_all["date"])

df_all.sort_values("Lpoly_expected_ml", ascending=False).head(10)[
    ["date", "Lpoly_expected_ml"]
]

,date,Lpoly_expected_ml
275,1970-01-01 00:00:00.020240518,157.102248
274,1970-01-01 00:00:00.020240517,124.132244
273,1970-01-01 00:00:00.020240516,77.451088
427,1970-01-01 00:00:00.020241105,75.524683
431,1970-01-01 00:00:00.020241109,64.535795
438,1970-01-01 00:00:00.020241122,60.840293
276,1970-01-01 00:00:00.020240519,51.421907
432,1970-01-01 00:00:00.020241110,48.022903
439,1970-01-01 00:00:00.020241123,46.004792
272,1970-01-01 00:00:00.020240515,43.817941
